In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
class State(TypedDict):
    request: str
    email: str
    action: Annotated[str, "The action to be performed by the agent"]
    is_approved: Annotated[bool, "Whether the action is approved by the user"]
    result: Annotated[str, "The result of the action"]

In [3]:
llm = ChatGroq(model='openai/gpt-oss-120b',temperature=0.1)
memory = InMemorySaver()

In [4]:
def create_email(state: State):
    print('create email')
    prompt = f"""
Create a short professional email based on this request:

{state['request']}

Return only the email body.
"""

    response = llm.invoke(prompt)

    return {"email": response.content}


def human_in_loop(state: State):
    print('Human in loop')
    answer = interrupt(
        f"Do you want to send the email?\n\n{state['email']}",
    )

    return {"is_approved": answer == "yes"}

def send_email(state: State):
    print('send email')
    if state['is_approved'] == False:
        return {"result": "Email cancelled."}

    print("EMAIL SENT:")
    print(state["email"])

    return {"result": "Email sent successfully."}

In [5]:
graph = StateGraph(State)

graph.add_node("create_email", create_email)
graph.add_node("human_in_loop", human_in_loop)
graph.add_node("send_email", send_email)

graph.add_edge(START, "create_email")
graph.add_edge("create_email", "human_in_loop")
graph.add_edge("human_in_loop", "send_email")
graph.add_edge("send_email", END)

workflow = graph.compile(checkpointer=memory)

In [6]:
CONFIG = { 'configurable' :{ 'thread_id': '1'} }
result = workflow.invoke(
    {"request": "Send an email to my manager saying I will be late today."},
    config=CONFIG
)

create email
Human in loop


In [24]:
result

{'request': 'Send an email to my manager saying I will be late today.',
 'email': 'Hi [Manager’s Name],\n\nI wanted to let you know that I will be arriving late to work today due to an unexpected personal matter. I expect to be in by [estimated arrival time] and will stay on top of any urgent tasks remotely in the meantime.\n\nThank you for your understanding.\n\nBest regards,  \n[Your Name]',
 '__interrupt__': [Interrupt(value='Do you want to send the email?\n\nHi [Manager’s Name],\n\nI wanted to let you know that I will be arriving late to work today due to an unexpected personal matter. I expect to be in by [estimated arrival time] and will stay on top of any urgent tasks remotely in the meantime.\n\nThank you for your understanding.\n\nBest regards,  \n[Your Name]', id='8bc11b4af5eb8bf97726ae486b0f7aef')]}

In [7]:
workflow.invoke(Command(resume="yes"), config=CONFIG)

Human in loop
send email
EMAIL SENT:
Hi [Manager’s Name],

I wanted to let you know that I will be arriving late to work today due to an unexpected personal matter. I expect to be in by [estimated arrival time] and will address any urgent tasks remotely in the meantime.

Thank you for your understanding.

Best regards,  
[Your Name]


{'request': 'Send an email to my manager saying I will be late today.',
 'email': 'Hi [Manager’s Name],\n\nI wanted to let you know that I will be arriving late to work today due to an unexpected personal matter. I expect to be in by [estimated arrival time] and will address any urgent tasks remotely in the meantime.\n\nThank you for your understanding.\n\nBest regards,  \n[Your Name]',
 'is_approved': True,
 'result': 'Email sent successfully.'}

In [28]:
CONFIG_2 = { 'configurable' :{ 'thread_id': '2'} }
result = workflow.invoke(
    {"request": "Send an email to my manager saying I will be late today."},
    config=CONFIG_2
)

In [ ]:
workflow.invoke(Command(resume="no"), config=CONFIG_2)

{'request': 'Send an email to my manager saying I will be late today.',
 'email': 'Subject: Late Arrival Today\n\nDear [Manager’s Name],\n\nI wanted to let you know that I will be arriving later than usual today due to [brief reason, e.g., a personal appointment]. I expect to be in the office by [estimated time].\n\nI apologize for any inconvenience this may cause and will ensure that all urgent tasks are addressed promptly upon my arrival.\n\nThank you for your understanding.\n\nBest regards,  \n[Your Name]',
 'is_approved': False,
 'result': 'Email cancelled.'}